# Homework 3 — Question 2

## (a) A generalization bound

Assume the test observations are i.i.d. and independent of the training data. Suppose the loss is bounded: $0\le L(y,f(x))\le B$. Let $f_1,\ldots,f_k$ be the models obtained from $k$ candidate hyperparameter settings. The settings are fixed before looking at the test set, and each candidate is fitted using training data only.

Define the true risk and test-set empirical risk by

$$
R(f)=E[L(Y,f(X))],\qquad
\widehat R_l(f)=\frac1l\sum_{i=1}^l L(Y_i,f(X_i)).
$$

Select $\widehat f=\arg\min_{f\in\{f_1,\ldots,f_k\}}\widehat R_l(f)$. For any $0<\delta<1$, with probability at least $1-\delta$,

$$
\boxed{\left|R(\widehat f)-\widehat R_l(\widehat f)\right|
\le B\sqrt{\frac{\log(2k/\delta)}{2l}}.}
$$

**Proof.** Condition on the training data (and any training randomness), so the $k$ candidate models are fixed. By Hoeffding's inequality, for each candidate $f_j$,

$$
P\left(\left|\widehat R_l(f_j)-R(f_j)\right|>\epsilon
\mid D_{\mathrm{train}}\right)
\le 2\exp\left(-\frac{2l\epsilon^2}{B^2}\right).
$$

Applying the union bound over all $k$ candidates gives

$$
P\left(\max_{1\le j\le k}\left|\widehat R_l(f_j)-R(f_j)\right|>\epsilon
\mid D_{\mathrm{train}}\right)
\le 2k\exp\left(-\frac{2l\epsilon^2}{B^2}\right).
$$

Set the right-hand side equal to $\delta$ and solve for $\epsilon$:

$$
\epsilon=B\sqrt{\frac{\log(2k/\delta)}{2l}}.
$$

The bound holds simultaneously for every candidate, so it also holds for the model selected using the test set. Since it holds conditional on any training data, it also holds unconditionally.

The training sample size $n$ does not appear explicitly because this bound concerns the accuracy of the held-out risk estimate, conditional on the fitted candidates. The training sample size still affects how good those candidates are.

Here $k$ means the number of candidate **configurations**. If instead $k$ counts separate hyperparameters, with $m_j$ choices for hyperparameter $j$, use $K=\prod_{j=1}^k m_j$ in place of $k$ in the bound. The number of separate hyperparameters alone does not specify the size of the search.

The bounded-loss assumption is necessary for this Hoeffding argument. For a loss in $[0,1]$, set $B=1$. For the wine experiment, the stated target range is $[1,10]$ and both selected model classes predict averages of training targets, so squared loss is bounded by $B=81$. A general unbounded squared loss would require additional tail assumptions for a comparable distribution-free bound.


## (b) Reusing the test set

When many configurations are compared, some may have low test error simply because of random variation in that particular test set. Selecting the lowest test error and reporting it as the final performance can therefore give an overly optimistic estimate.

The bound in part (a) becomes weaker as $\log k/l$ increases. Thus, a large search and a small evaluation set make selection effects more concerning. The number of tried configurations matters, not just the number of hyperparameter names.

A separate validation set can be used to select the configuration. The test set is then used only after tuning is complete to evaluate the selected model. This avoids using the same observations for both selection and final evaluation.


## (c) White wine experiment

I use `RandomForestRegressor` and `KNeighborsRegressor`. The file is semicolon-separated and contains 4,898 observations, 11 input features, and no missing values. `quality` is the target, treated as a regression outcome. I use the original dataset without removing duplicate rows.

Using random seed 42, I first reserve 20% for testing, then reserve 12.5% of the remaining observations for validation. This gives approximately 70% training, 10% validation, and 20% test:

| Split | Observations |
|---|---:|
| Training | 3,428 |
| Validation | 490 |
| Test | 980 |

I select parameters by minimizing validation MSE. I use an explicit `ParameterGrid` loop rather than the default cross-validation procedure in `GridSearchCV`, so selection uses exactly the requested validation split. The test set is evaluated only after both searches are complete.

### Model parameters and search grids

For random forest, I vary:

| Parameter | Meaning | Values |
|---|---|---|
| `n_estimators` | Number of trees | 20, 40, 60 |
| `max_depth` | Maximum tree depth | 3, 5, 8, 12, None |
| `min_samples_leaf` | Minimum observations per leaf | 1, 2, 4, 8 |
| `max_features` | Fraction of features considered at a split | 0.5, 0.75, 1.0 |
| `max_samples` | Bootstrap sample size as a fraction of training size | 0.5, 0.75, 1.0 |

This gives $3\times5\times4\times3\times3=540$ configurations, exceeding the 490 validation observations. I fix `random_state=42` and use the default squared-error criterion and bootstrapping.

For KNN, I vary:

| Parameter | Meaning | Values |
|---|---|---|
| `n_neighbors` | Number of neighbors | Every integer from 1 to 150 |
| `weights` | How neighbors contribute to predictions | uniform, distance |
| `p` | Distance metric: 1 for Manhattan, 2 for Euclidean | 1, 2 |

This gives $150\times2\times2=600$ configurations, also exceeding 490. I standardize the inputs for KNN, fitting the scaler on training data only. Random forest uses the original inputs.

Parameter definitions were checked against the official documentation:

- [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)


### Imports

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


### Read data and split 70% / 10% / 20%

In [ ]:
# This file uses semicolons, not commas.
data = pd.read_csv("winequality-white-1.csv", sep=";")
x = data.drop(columns="quality")
y = data["quality"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.125, random_state=42
)

print("Data shape:", data.shape)
print("Missing values:", int(data.isna().sum().sum()))
print("Train / validation / test:", len(y_train), len(y_val), len(y_test))

Data shape: (4898, 12)
Missing values: 0
Train / validation / test: 3428 490 980


### Parameter grids

In [3]:
rf_grid = {
    "n_estimators": [20, 40, 60],
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [0.5, 0.75, 1.0],
    "max_samples": [0.5, 0.75, 1.0]
}

knn_grid = {
    "n_neighbors": list(range(1, 151)),
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

assert len(ParameterGrid(rf_grid)) > len(y_val)
assert len(ParameterGrid(knn_grid)) > len(y_val)
print("Random forest configurations:", len(ParameterGrid(rf_grid)))
print("KNN configurations:", len(ParameterGrid(knn_grid)))



Random forest configurations: 540
KNN configurations: 600


### Random forest grid search

In [4]:
best_rf_loss = float("inf")
rf_rows = []

for i, params in enumerate(ParameterGrid(rf_grid)):
    model = RandomForestRegressor(
        **params, random_state=42, n_jobs=2
    )
    model.fit(x_train, y_train)
    pred = model.predict(x_val)
    loss = mean_squared_error(y_val, pred)
    rf_rows.append({**params, "Validation MSE": loss})

    if loss < best_rf_loss:
        best_rf_loss = loss
        best_rf = model
        best_rf_params = params.copy()

    if (i + 1) % 100 == 0:
        print(f"Random forest: {i + 1} configurations finished")

pd.DataFrame(rf_rows).to_csv("rf_search.csv", index=False)
print("Best random forest parameters:", best_rf_params)
print("Best random forest validation MSE:", round(best_rf_loss, 6))



Random forest: 100 configurations finished
Random forest: 200 configurations finished
Random forest: 300 configurations finished
Random forest: 400 configurations finished
Random forest: 500 configurations finished
Best random forest parameters: {'max_depth': None, 'max_features': 0.75, 'max_samples': 1.0, 'min_samples_leaf': 1, 'n_estimators': 40}
Best random forest validation MSE: 0.310866


### Standardize inputs for KNN

In [ ]:
# The scaler learns its mean and standard deviation from training data only.
scaler = StandardScaler()
a_train = scaler.fit_transform(x_train)
a_val = scaler.transform(x_val)

### KNN grid search

In [ ]:
best_knn_loss = float("inf")
knn_rows = []

for i, params in enumerate(ParameterGrid(knn_grid)):
    model = KNeighborsRegressor(**params, n_jobs=2)
    model.fit(a_train, y_train)
    pred = model.predict(a_val)
    loss = mean_squared_error(y_val, pred)
    knn_rows.append({**params, "Validation MSE": loss})

    if loss < best_knn_loss:
        best_knn_loss = loss
        best_knn = model
        best_knn_params = params.copy()

    if (i + 1) % 100 == 0:
        print(f"KNN: {i + 1} configurations finished")

pd.DataFrame(knn_rows).to_csv("knn_search.csv", index=False)
print("Best KNN parameters:", best_knn_params)
print("Best KNN validation MSE:", round(best_knn_loss, 6))

KNN: 100 configurations finished
KNN: 200 configurations finished
KNN: 300 configurations finished
KNN: 400 configurations finished
KNN: 500 configurations finished
KNN: 600 configurations finished
Best KNN parameters: {'n_neighbors': 25, 'p': 1, 'weights': 'distance'}
Best KNN validation MSE: 0.298739


### Final test evaluation after both searches are complete

In [7]:
a_test = scaler.transform(x_test)
rf_pred = best_rf.predict(x_test)
knn_pred = best_knn.predict(a_test)

results = pd.DataFrame([
    {"Model": "Random forest", "Configurations": len(rf_rows),
     "Validation MSE": best_rf_loss,
     "Test MSE": mean_squared_error(y_test, rf_pred)},
    {"Model": "KNN", "Configurations": len(knn_rows),
     "Validation MSE": best_knn_loss,
     "Test MSE": mean_squared_error(y_test, knn_pred)}
])
results["Test minus validation"] = results["Test MSE"] - results["Validation MSE"]
print(results.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
results.to_csv("results.csv", index=False)

# These are the selected training-fitted models, not train+validation refits.
with open("best_params.json", "w") as f:
    json.dump({"Random forest": best_rf_params, "KNN": best_knn_params}, f, indent=2)

# Keep row indices to make the test predictions easy to verify.
pd.DataFrame({
    "Row": y_test.index, "quality": y_test.to_numpy(),
    "Random forest": rf_pred, "KNN": knn_pred
}).to_csv("test_results.csv", index=False)


        Model  Configurations  Validation MSE  Test MSE  Test minus validation
Random forest             540        0.310866  0.358765               0.047899
          KNN             600        0.298739  0.368550               0.069811


### Selected parameters

**Random forest**

```python
{'max_depth': None, 'max_features': 0.75, 'max_samples': 1.0, 'min_samples_leaf': 1, 'n_estimators': 40}
```

**KNN**

```python
{'n_neighbors': 25, 'p': 1, 'weights': 'distance'}
```

### Results

| Model | Configurations | Validation MSE | Test MSE | Test minus validation |
|---|---:|---:|---:|---:|
| Random forest | 540 | 0.310866 | 0.358765 | 0.047899 |
| KNN | 600 | 0.298739 | 0.368550 | 0.069811 |

For Random forest, test MSE is 0.047899 higher than validation MSE. For KNN, test MSE is 0.069811 higher than validation MSE. 

Validation MSE is used repeatedly to choose the configuration, so the selected validation error can be optimistic. Test MSE is a separate estimate of the selected model's performance. The test error is not guaranteed to exceed validation error in every split because both are affected by sampling variation, and the candidate models can be highly correlated.

I evaluate the selected models as fitted on the training set; I do not refit on training plus validation before making this comparison. Predictions are not rounded because the task uses regression MSE.

**Random forest** has the lower test MSE in this experiment.
